# Neptune Exploration via SSH Tunnel

This notebook demonstrates connecting to the DIA Neptune cluster through
a bastion host via EC2 Instance Connect Endpoint (EICE).

## Why a bastion + SSH tunnel (not EICE directly)

I initially attempted to use EICE's `open-tunnel` command to connect
directly to Neptune's private IP on port 8182. This failed because EICE
only supports remote ports 22 (SSH) and 3389 (RDP) — confirmed via live
testing against the deployed cluster:

```
InvalidParameter: The specified RemotePort is not valid.
Specify either 22 or 3389 as the RemotePort and retry your request.
```

The solution: use EICE to SSH into a bastion (port 22, which EICE does
support), then use SSH's own local port-forwarding (`-L`) to relay
traffic from localhost:8182 to Neptune's endpoint inside the VPC. The
bastion resolves Neptune's private DNS hostname server-side (inside the
VPC), which also sidesteps a separate issue where Neptune's cluster
endpoint does not resolve via public DNS from outside the VPC. Using a bastion
also opens up the possibility of running workloads on the bastion itself if needed and triggering via the EICE -> Bastion route.

## Prerequisites

- Neptune cluster + bastion deployed (`cdk deploy dia-networking-dev dia-bastion-dev dia-neptune-dev`)
- AWS CLI v2 installed and authenticated to the dev account
- Tunnel running: `./scripts/neptune-tunnel.sh dev` in a separate terminal

## How it works

1. Authenticate yourself in your terminal with a specific AWS profile.
2. Run `./scripts/neptune-tunnel.sh dev` — SSH-tunnels through the bastion via EICE. No admin/sudo access needed on your local machine.
3. This notebook uses `LocalNeptuneClient` to query Neptune at the real endpoint hostname — DNS resolution for that hostname is patched at the Python-process level (inside `LocalNeptuneClient`), so no `/etc/hosts` modification is needed either.
4. When you Ctrl+C the tunnel script, the SSH session and port-forward simply close — there's no cleanup step required.

The tunnel auto-expires after 1 hour (EICE hard limit). If it drops,
just re-run `./scripts/neptune-tunnel.sh dev`.

## Currently the bastion has no S3 access

The bastion currently has **no IAM instance profile** — it's purely an SSH
relay for port-forwarding, and makes no AWS API calls itself. This is
different from a real person: when you query Neptune through the
tunnel, those calls are authenticated as *your* IAM identity (forwarded
through the SSH session), not the bastion's.

**Future work:** if we want to run data-loading scripts on the bastion
itself (e.g. reading extracted graph data from the `graph-validated` S3
bucket and writing it to Neptune), the bastion would need:

1. **Its own IAM role** granting S3 read access on the relevant pipeline
   buckets, plus `neptune-db:*` (or narrower) permissions on the cluster
2. **A network path to S3** — the shared VPC's existing S3 interface
   endpoint only allows traffic from a specific load balancer's security
   group, not arbitrary VPC resources (including our bastion).

In [1]:
from dia.clients.neptune import LocalNeptuneClient

# Connect to Neptune via the local SSH tunnel.
# - endpoint: from `dia-neptune-dev` stack outputs (update if cluster is recreated)
neptune = LocalNeptuneClient(
    endpoint="dia-neptune-dev.cluster-c3a42mmuka2e.eu-west-2.neptune.amazonaws.com",
    profile_name="default",
)

# Verify connectivity — should return [] on an empty cluster (expected)
neptune.query("MATCH (n) RETURN labels(n) AS labels, count(n) AS count")

[]

In [2]:
# Example: browse all nodes (once data is loaded)
neptune.query("MATCH (n) RETURN n LIMIT 5")

[]

In [3]:
# Example: find relationships for a specific entity
# neptune.query("MATCH (a)-[r]->(b) WHERE a.name = 'Some Entity' RETURN a, r, b LIMIT 10")

In [ ]:
import os

AWS_PROFILE = "default"
AWS_REGION = "eu-west-2"
EXTRACTION_MODEL = "eu.anthropic.claude-sonnet-4-6"
RESPONSE_MODEL = "eu.anthropic.claude-sonnet-4-6"
EMBEDDINGS_MODEL = "amazon.titan-embed-text-v2:0"

os.environ["AWS_REGION"] = AWS_REGION
os.environ["EXTRACTION_MODEL"] = EXTRACTION_MODEL
os.environ["RESPONSE_MODEL"] = RESPONSE_MODEL
os.environ["EMBEDDINGS_MODEL"] = EMBEDDINGS_MODEL
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["aws_profile"] = AWS_PROFILE

AOSS_ENDPOINT = "dummy//"  # replace with actual
NEPTUNE_ENPOINT = "dummy//"  # replace with actual

from graphrag_toolkit.lexical_graph import LexicalGraphQueryEngine, set_logging_config  # noqa: E402
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory, VectorStoreFactory  # noqa: E402

set_logging_config("INFO")

with (
    GraphStoreFactory.for_graph_store(NEPTUNE_ENPOINT) as graph_store,
    VectorStoreFactory.for_vector_store(f"aoss://{AOSS_ENDPOINT}") as vector_store,
):
    query_engine = LexicalGraphQueryEngine.for_traversal_based_search(graph_store, vector_store, streaming=True)

    response = query_engine.query("What are the differences between Neptune Database and Neptune Analytics?")

print(f"""{response.print_response_stream()}

retrieve_ms: {int(response.metadata["retrieve_ms"])}
answer_ms  : {int(response.metadata["answer_ms"])}
total_ms   : {int(response.metadata["total_ms"])}
""")

The search results are empty, so I'm unable to answer your question about the differences between Neptune Database and Neptune Analytics based on the provided search results.

To get accurate information about this topic, I'd recommend:
1. Visiting the **AWS Neptune documentation** at [docs.aws.amazon.com](https://docs.aws.amazon.com)
2. Checking the **AWS Neptune product page** for a feature comparison
3. Looking at the **AWS blog** for announcements and detailed comparisonsNone

retrieve_ms: 3566
answer_ms  : 775
total_ms   : 4342

